<a href="https://colab.research.google.com/github/katerina-tech/WiDS-Global-Datathon-2026-Erope-Team/blob/main/WiDS_2026_%7C_Get_Started_Europe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

widsworldwide_globaldathon26_path = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
hmnshudhmn24_wids_global_datathon_2026_path = kagglehub.notebook_output_download('hmnshudhmn24/wids-global-datathon-2026')
belbino_wildfire_scikitlearn_default_1_path = kagglehub.model_download('belbino/wildfire/ScikitLearn/default/1')

print('Data source import complete.')


 # **WiDS Global Datathon 2026**

# IMPORT LIBRARY

In [ ]:
import cloudpickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

```python
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
```

# DATA PREPROCESSING

## LOAD DATA

In [ ]:
meta_data_df = pd.read_csv("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv")
train_df = pd.read_csv("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv")


## EDA

In [ ]:
# 1. How many fires actually hit vs censored?
plt.figure(figsize=(6, 4))
sns.countplot(x='event', data=train_df, hue='event', palette='viridis', legend=False)
plt.title('Target Distribution: Hits (1) vs. Censored (0)')
plt.show()
# 2. When do the hits happen?
plt.figure(figsize=(8, 5))
sns.histplot(train_df[train_df['event'] == 1]['time_to_hit_hours'], bins=15, color='orange', kde=True)
plt.axvline(24, color='red', linestyle='--', label='24h Mark')
plt.title('Distribution of Time-to-Hit (Only for Event=1)')
plt.legend()
plt.show()

In [ ]:

killer_features = [
    'closing_speed_m_per_h',
    'dist_min_ci_0_5h',
    'area_growth_rate_ha_per_h',
    'alignment_abs',
    'num_perimeters_0_5h'
]

comparison_df = train_df.groupby('event')[killer_features].agg(['mean', 'median']).T
train_df['speed_dist_ratio'] = train_df['closing_speed_m_per_h'] / (train_df['dist_min_ci_0_5h'] + 1)

hits_only = train_df[train_df['event'] == 1]
time_correlations = hits_only[killer_features + ['time_to_hit_hours']].corr()['time_to_hit_hours'].sort_values()

display(train_df['event'].value_counts())

print("\n--- Summary of Time-to-Hit (Hits Only) ---\n")
display(train_df[train_df['event'] == 1]['time_to_hit_hours'].describe())

print("\n--- Numerical Comparison: No-Hit (0) vs Hit (1) ---\n")
display(comparison_df)

print("\n--- Speed/Distance Ratio Statistics ---\n")
display(train_df.groupby('event')['speed_dist_ratio'].describe())

print("\n--- Correlations with Time-to-Hit (Numerical) ---\n")
print(time_correlations)


## DATA PREPARATION

### TARGET COLUMNS

```python
# The competition horizons
horizons = [12, 24, 48, 72]

for h in horizons:
    train_df[f'target_{h}h'] = ((train_df['event'] == 1) & (train_df['time_to_hit_hours'] <= h)).astype(int)

# Check the imbalance for each
print("--- Positive Cases (Hits) per Horizon ---")
for h in horizons:
    count = train_df[f'target_{h}h'].sum()
    percentage = (count / len(train_df)) * 100
    print(f"{h}h Horizon: {count} hits ({percentage:.1f}%)")
```

--- Positive Cases (Hits) per Horizon ---
- 12h Horizon: 49 hits (22.2%)
- 24h Horizon: 63 hits (28.5%)
- 48h Horizon: 66 hits (29.9%)
- 72h Horizon: 69 hits (31.2%)

#### FEATURES & TARGET

```python
drop_cols = ['time_to_hit_hours', 'speed_dist_ratio',
    'event', 'target_12h', 'target_24h',
    'target_48h', 'target_72h', 'fold']

base_features = [col for col in train_df.columns if col not in drop_cols]

X=train_df[base_features]
y=train_df[['target_12h', 'target_24h','target_48h', 'target_72h']]
```

## FEATURE ENGINEERING

```python
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        if 'event_id' in X.columns:
            X = X.drop(columns=['event_id'])
        
    
        X['safe_closing_speed'] = X['closing_speed_m_per_h'].clip(lower=0)
        
        # Effective Speed
        X['effective_speed'] = (X['closing_speed_m_per_h'] * X['alignment_abs']).clip(lower=0)

        # Arrival Estimates
        X['simple_eta'] = X['dist_min_ci_0_5h'] / (X['safe_closing_speed'] + 1e-5)
        X['log_eta'] = np.log1p(X['dist_min_ci_0_5h'] / (X['effective_speed'] + 1))
        
        # Threat and Ratios
        X['speed_dist_ratio'] = X['safe_closing_speed'] / (X['dist_min_ci_0_5h'] + 1)
        X['threat_score'] = (X['effective_speed']) / (np.log1p(X['dist_min_ci_0_5h']) + 1)
        
        # Growth and Momentum
        X['growth_intensity'] = X['area_growth_abs_0_5h'] / (X['area_first_ha'] + 1)
        X['growth_momentum'] = X['area_growth_rate_ha_per_h'] * X['alignment_cos']
        
        # Temporal & Quality Features
        X['mapping_frequency'] = X['num_perimeters_0_5h'] / (X['dt_first_last_0_5h'] + 0.1)
        X['hour_sin'] = np.sin(2 * np.pi * X['event_start_hour'] / 24)
        X['hour_cos'] = np.cos(2 * np.pi * X['event_start_hour'] / 24)
        
        return X
```

# MODEL

```python
xgb_params = {
    'n_estimators': 200,
    'max_depth': 3,
    'learning_rate': 0.05,
    'random_state': 42
}

base_xgb = xgb.XGBClassifier(**xgb_params)

calibrated_xgb = CalibratedClassifierCV(base_xgb, cv=5)

Model = MultiOutputClassifier(calibrated_xgb)
```

# PIPELINE

```python
pipeline = Pipeline([
    ('engineer', FeatureEngineer()),
    ('scaler', StandardScaler()),
    ('model', Model)
])
```

# TRAINING

```python
pipeline.fit(X,y)
```

# SAVE MODEL

```python

model_filename = 'wildfire_xgb_1.pkl'

with open(model_filename, 'wb') as f:
    cloudpickle.dump(pipeline, f)
```

# LOAD MODEL

In [ ]:
file ="/kaggle/input/models/belbino/wildfire/scikitlearn/default/1/wildfire_xgb_1.pkl"

with open (file,'rb')as f:
    Model =cloudpickle.load(f)

Model

# PREDICTION

In [ ]:
probs_list = Model.predict_proba(test_df)


submission = pd.DataFrame({
    'event_id': test_df['event_id'],
    'prob_12h': probs_list[0][:, 1],
    'prob_24h': probs_list[1][:, 1],
    'prob_48h': probs_list[2][:, 1],
    'prob_72h': probs_list[3][:, 1]
}, index=test_df.index)

prob_cols = ['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']

submission[prob_cols] = np.maximum.accumulate(submission[prob_cols].values, axis=1)

# SUBMISSION

In [ ]:
submission.to_csv('submission.csv', index=False)

In [ ]:
submission